# Wariant 14: Rehabilitacja poudarowa (RL)

Cel: implementacja MDP i Q-learning dla wariantu 14 oraz krótka interpretacja wyników.

To jest symulacja dydaktyczna (nie narzędzie kliniczne).

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

# ===== 1) Definicja MDP (wariant 14) =====
STATES = ["SevereDisability", "ModerateDisability", "Independent"]
ACTIONS = ["NoRehab", "StandardRehab", "IntensiveRehab"]

S = {name: i for i, name in enumerate(STATES)}
A = {name: i for i, name in enumerate(ACTIONS)}

gamma = 0.95

# P(s' | s, a)
transitions = {
    (S["SevereDisability"], A["NoRehab"]): [
        (S["SevereDisability"], 0.75), (S["ModerateDisability"], 0.22), (S["Independent"], 0.03)
    ],
    (S["SevereDisability"], A["StandardRehab"]): [
        (S["ModerateDisability"], 0.55), (S["SevereDisability"], 0.40), (S["Independent"], 0.05)
    ],
    (S["SevereDisability"], A["IntensiveRehab"]): [
        (S["ModerateDisability"], 0.60), (S["Independent"], 0.20), (S["SevereDisability"], 0.20)
    ],

    (S["ModerateDisability"], A["NoRehab"]): [
        (S["ModerateDisability"], 0.70), (S["SevereDisability"], 0.10), (S["Independent"], 0.20)
    ],
    (S["ModerateDisability"], A["StandardRehab"]): [
        (S["Independent"], 0.40), (S["ModerateDisability"], 0.55), (S["SevereDisability"], 0.05)
    ],
    (S["ModerateDisability"], A["IntensiveRehab"]): [
        (S["Independent"], 0.55), (S["ModerateDisability"], 0.40), (S["SevereDisability"], 0.05)
    ],

    (S["Independent"], A["NoRehab"]): [
        (S["Independent"], 0.85), (S["ModerateDisability"], 0.13), (S["SevereDisability"], 0.02)
    ],
    (S["Independent"], A["StandardRehab"]): [
        (S["Independent"], 0.88), (S["ModerateDisability"], 0.11), (S["SevereDisability"], 0.01)
    ],
    (S["Independent"], A["IntensiveRehab"]): [
        (S["Independent"], 0.90), (S["ModerateDisability"], 0.09), (S["SevereDisability"], 0.01)
    ],
}

# Nagrody: +10 za przejście do Independent, -5 za przejście do SevereDisability
# Koszty: NoRehab=0, StandardRehab=-2, IntensiveRehab=-4
action_cost = {
    A["NoRehab"]: 0.0,
    A["StandardRehab"]: -2.0,
    A["IntensiveRehab"]: -4.0,
}

def reward_fn(next_state, action):
    reward = 0.0
    if next_state == S["Independent"]:
        reward += 10.0
    if next_state == S["SevereDisability"]:
        reward -= 5.0
    reward += action_cost[action]
    return reward

class RehabEnv:
    def __init__(self, max_steps=20):
        self.max_steps = max_steps
        self.state = S["SevereDisability"]
        self.t = 0

    def reset(self, start_state="SevereDisability"):
        self.state = S[start_state]
        self.t = 0
        return self.state

    def step(self, action):
        self.t += 1
        probs = transitions[(self.state, action)]

        r = np.random.rand()
        cum = 0.0
        next_state = self.state
        for ns, p in probs:
            cum += p
            if r <= cum:
                next_state = ns
                break

        reward = reward_fn(next_state, action)
        self.state = next_state
        done = self.t >= self.max_steps
        return next_state, reward, done

# ===== 2) Q-learning =====
def epsilon_greedy(Q, s, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(Q.shape[1])
    return int(np.argmax(Q[s]))

def q_learning(env, episodes=3000, alpha=0.1, gamma=0.95, epsilon=0.3, epsilon_decay=0.999, epsilon_min=0.05):
    Q = np.zeros((len(STATES), len(ACTIONS)))
    rewards = []

    for _ in range(episodes):
        s = env.reset(start_state="SevereDisability")
        total = 0.0

        while True:
            a = epsilon_greedy(Q, s, epsilon)
            s2, r, done = env.step(a)
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])
            total += r
            s = s2
            if done:
                break

        rewards.append(total)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return Q, rewards

env = RehabEnv(max_steps=20)
Q, rewards = q_learning(env)

# ===== 3) Polityka i wyjaśnienie =====
policy = {STATES[s]: ACTIONS[int(np.argmax(Q[s]))] for s in range(len(STATES))}
q_table = pd.DataFrame(Q, index=STATES, columns=ACTIONS)

print("Polityka (wariant 14):")
for state_name, action_name in policy.items():
    print(f"{state_name:20s} -> {action_name}")

print("\nTabela Q:")
display(q_table)

print("\nSrednia nagroda (ostatnie 200 epizodow):", np.mean(rewards[-200:]))

# Explainable RL: analiza Q(s,a) i roznicy DeltaQ
def explain_action(Q, state_name):
    s_idx = S[state_name]
    vals = Q[s_idx]
    df = pd.DataFrame({"akcja": ACTIONS, "Q(s,a)": vals}).sort_values("Q(s,a)", ascending=False).reset_index(drop=True)
    best = df.loc[0, "akcja"]
    delta_q = float(df.loc[0, "Q(s,a)"] - df.loc[1, "Q(s,a)"])
    return df, best, delta_q

for st in STATES:
    df, best, delta_q = explain_action(Q, st)
    print(f"\nStan: {st}")
    display(df)
    print(f"Wybrana akcja: {best}; DeltaQ(best-second): {delta_q:.3f}")

Polityka (wariant 14):
SevereDisability     -> IntensiveRehab
ModerateDisability   -> IntensiveRehab
Independent          -> NoRehab

Tabela Q:


,NoRehab,StandardRehab,IntensiveRehab
SevereDisability,97.070306,100.245763,123.282373
ModerateDisability,109.350493,115.639552,129.375281
Independent,142.710521,126.690145,129.919616



Srednia nagroda (ostatnie 200 epizodow): 107.155

Stan: SevereDisability


,akcja,"Q(s,a)"
0,IntensiveRehab,123.282373
1,StandardRehab,100.245763
2,NoRehab,97.070306


Wybrana akcja: IntensiveRehab; DeltaQ(best-second): 23.037

Stan: ModerateDisability


,akcja,"Q(s,a)"
0,IntensiveRehab,129.375281
1,StandardRehab,115.639552
2,NoRehab,109.350493


Wybrana akcja: IntensiveRehab; DeltaQ(best-second): 13.736

Stan: Independent


,akcja,"Q(s,a)"
0,NoRehab,142.710521
1,IntensiveRehab,129.919616
2,StandardRehab,126.690145


Wybrana akcja: NoRehab; DeltaQ(best-second): 12.791


## Krótkie wyjaśnienie

- MDP opisuje stany pacjenta, możliwe decyzje terapeutyczne i losowe przejścia między stanami.
- Q-learning uczy się wartości `Q(s,a)`, czyli opłacalności akcji w danym stanie.
- Polityka wybiera akcję z najwyższą wartością Q.
- Explainable RL: porównujemy wszystkie `Q(s,a)` i różnicę `DeltaQ` między najlepszą i drugą akcją.

## Krótkie wnioski

1. W stanie ciężkim zwykle opłaca się IntensiveRehab, bo częściej prowadzi do poprawy mimo większego kosztu.
2. W stanie umiarkowanym decyzja zależy od balansu kosztu i szansy przejścia do Independent.
3. W stanie Independent często wystarcza NoRehab lub StandardRehab, bo celem jest utrzymanie dobrego stanu przy niższym koszcie.
4. Małe `DeltaQ` oznacza niepewność decyzji (alternatywna akcja jest prawie tak dobra).